# WealthPilot AI: Autonomous Portfolio Rebalancing Cycle Demo

This interactive walkthrough demonstrates an end-to-end autonomous rebalancing cycle:
1. **Vectorized Drift Monitoring**: Scanning a portfolio for strategic asset allocation deviations.
2. **Three-Tier Trigger Consolidation**: Evaluating threshold breaches and queue prioritization.
3. **Convex QP Optimization (CVXPY)**: Solving tracking error variance minimization under hard SEBI constraints.
4. **Tax Management & Lot Selection**: Executing HIFO lot depletion, LTCG exemptions (Section 112A), and wash-sale avoidance.
5. **Explainable AI (XAI)**: Generating multi-audience narratives and Tree SHAP feature attributions.

In [1]:
import datetime
import numpy as np
import pandas as pd

from src.monitoring.drift_calculator import DriftCalculator
from src.triggers.trigger_consolidator import TriggerConsolidator, TriggerEvent
from src.optimisation.portfolio_optimiser import PortfolioOptimiser
from src.optimisation.tax_lot_manager import TaxLotManager, TaxLot
from src.optimisation.tax_optimiser import TaxOptimiser
from src.optimisation import TradeOrder
from src.explainability.explanation_generator import ExplanationGenerator
from src.compliance.explainability_scorecard import ExplainabilityScorecard

print("WealthPilot AI core modules loaded successfully.")

## Step 1: Vectorized Drift Monitoring
We evaluate an active portfolio (`PORT-00104`, Balanced Risk Profile, ₹50,00,000 AUM) where an equity rally has pushed large-cap stocks overweight.

In [2]:
assets = ["NIFTY_50_EQUITY", "G_SEC_BONDS", "CORP_BONDS", "GOLD_ETF", "LIQUID_CASH"]
curr_w = np.array([[0.55, 0.25, 0.10, 0.05, 0.05]])  # 55% Equity vs 40% Target
targ_w = np.array([[0.40, 0.30, 0.15, 0.10, 0.05]])

calc = DriftCalculator(asset_names=assets)
metrics = calc.calculate_drift_matrix(curr_w, targ_w)

print(f"Signed Drift (%): {metrics['signed_drift'][0] * 100}")
print(f"Max Absolute Drift: {metrics['max_drift'][0]:.2%}")
print(f"Sum of Absolute Deviations (SAD): {metrics['sad'][0]:.2%}")

## Step 2: Three-Tier Trigger Consolidation
The 15.0% equity drift breaches the 5.0% tolerance band, generating a prioritized rebalance task.

In [3]:
consolidator = TriggerConsolidator()
trigger = TriggerEvent(
    portfolio_id="PORT-00104",
    trigger_type="THRESHOLD",
    severity_score=3.0,
    details={"max_drift": metrics['max_drift'][0], "drift_asset": "NIFTY_50_EQUITY"}
)

queue = consolidator.consolidate_triggers([trigger])
print(f"Queued Task: {queue[0].portfolio_id} | Priority Score: {queue[0].priority_score:.2f} | Category: {queue[0].trigger_type}")

## Step 3: Convex Quadratic Programming Optimization (CVXPY)
We formulate a constrained QP minimizing tracking error variance under SEBI concentration caps and turnover limits.

In [4]:
optimiser = PortfolioOptimiser(asset_names=assets)
opt_res = optimiser.optimize_allocation(
    current_weights=dict(zip(assets, curr_w[0])),
    target_weights=dict(zip(assets, targ_w[0])),
    turnover_budget=0.15,
    min_cash_buffer=0.02,
)

print("Optimal Rebalanced Weights:")
for k, v in opt_res["optimal_weights"].items():
    print(f"  {k:20s}: {v:.2%}")
print(f"Predicted Tracking Error: {opt_res['predicted_tracking_error']:.4f}")
print(f"One-Way Turnover: {opt_res['turnover']:.2%}")

## Step 4: Tax Management & Lot Selection
Applying HIFO depletion to trim equity while shielding capital gains under Section 112A (₹1.25 Lakh exemption).

In [5]:
lot_mgr = TaxLotManager()
lot_mgr.add_lot(TaxLot(
    lot_id="LOT-001",
    portfolio_id="PORT-00104",
    asset_class="NIFTY_50_EQUITY",
    quantity=1000.0,
    cost_per_share=220.0,
    current_price=250.0,
    acquisition_date=datetime.date(2024, 1, 10)
))

tax_opt = TaxOptimiser()
sell_order = TradeOrder(
    portfolio_id="PORT-00104",
    asset_class="NIFTY_50_EQUITY",
    action="SELL",
    target_units=600.0,
    estimated_price=250.0,
    trade_value_inr=150000.0
)

tax_plan = tax_opt.evaluate_trade_plan_taxes(
    portfolio_id="PORT-00104",
    orders=[sell_order],
    lot_manager=lot_mgr,
    strategy="TAX_MINIMIZER"
)

print(f"Realized LTCG: INR {tax_plan['tax_summary']['net_taxable_ltcg']:.2f}")
print(f"Section 112A Exemption Claimed: INR {tax_plan['tax_summary']['ltcg_exemption_applied']:.2f}")
print(f"Net Tax Liability: INR {tax_plan['total_estimated_tax_inr']:.2f}")

## Step 5: Multi-Audience Explainable AI & Scorecard Verification
Synthesizing audience-specific narratives and scoring them with the institutional Explainability Scorecard.

In [6]:
scorecard = ExplainabilityScorecard()
sample_packet = {
    "decision_id": "DEC-DEMO-001",
    "client_explanation": {
        "headline": "Portfolio Rebalanced to Safeguard Equity Profits",
        "narrative": "We safely trimmed your equity gains after stocks grew to 55% of your portfolio, locking profits into government debt.",
        "goal_relevance": "Maintains your agreed balanced retirement risk profile.",
        "cost_transparency": {"total_estimated_costs_inr": 250.0, "current_equity_pct": 55.0},
        "readability_grade": 7.4,
        "word_count": 90,
    },
    "advisor_dossier": {
        "executive_summary": "Convex QP rebalanced 15% equity drift into sovereign bonds. Tracking error reduced to 0.45%.",
        "drift_metrics": {"equity_drift": 0.15},
        "counterfactual_alternatives": ["Hold allocation and incur uncompensated tracking error penalty"],
        "word_count": 195,
    },
    "compliance_audit": {
        "sebi_compliance_certified": True,
        "regulatory_circular": "SEBI/HO/MRD/DOP1/CIR/P/2024/69",
        "constraint_verification_matrix": {"single_issuer_ok": True, "sector_cap_ok": True},
        "digital_signature_hash": "c7b1a2f3e4d586901234567890abcdefc7b1a2f3e4d586901234567890abcdef",
    }
}

score = scorecard.score_explanation_packet(sample_packet, actual_trade_context={"total_costs_inr": 250.0, "current_equity_pct": 55.0})
print(f"Scorecard Composite Grade: {score.grade} ({score.composite_score}/100 pts)")
print(f"Compliance Gate Passed: {score.is_compliant}")
print("Cycle completed successfully!")